# Wholebody Pose Estimation using SAPIENS Model

This notebook predicts wholebody poses (body + hands + face) using the SAPIENS model.
It processes videos from the inputs/{data_collection} directory and outputs wholebody keypoints.

In [1]:
import os
import cv2
import numpy as np
import mmcv
import json
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
from mmpose.structures import split_instances

from helpers.predictors import *
from helpers.definitions import *

Device: cuda
Torch compile enabled: True


## Configuration

Set up the data collection path and other configuration parameters.

In [2]:
# Configuration
data_collection = 'cha/cha_short'  # Change this to match your data collection
show = True  # Show visualizations
draw_bbox = True  # Draw bounding boxes
original_resolution = (3840, 2160)  # Original video resolution
resolution = (1920, 1080)  # Resolution for processing
kpt_thresh = 0.3  # Keypoint confidence threshold

# Create output directories
pose_output_dir = f'pose_outputs/{data_collection}'
vis_output_dir = f'vis_dir/{data_collection}'

os.makedirs(pose_output_dir, exist_ok=True)
os.makedirs(vis_output_dir, exist_ok=True)

# Input data directory
data_dir = f'inputs/{data_collection}'

# Ensure data directory exists
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"Data directory not found: {data_dir}")

## Initialize Models

Initialize the SAPIENS model and associated visualizer.

In [3]:
# Initialize models with SAPIENS for wholebody pose estimation
initialize_models(use_tam=True, use_wholebody=True)

# Initialize YOLO model for person detection
model_person = YOLO('checkpoints/yolo/yolo11l.pt')
model_person.conf = 0.5  # Confidence threshold

Loads checkpoint by local backend from path: checkpoints/hands/detection/cascade_rcnn_x101_64x4d_fpn_20e_onehand10k-dac19597_20201030.pth
Loads checkpoint by local backend from path: checkpoints/body/detection/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth
Disable torch compile due to unsupported GPU.
Loads checkpoint by local backend from path: checkpoints/hands/pose/rtmpose-m_simcc-hand5_pt-aic-coco_210e-256x256-74fb594_20230320.pth
Loads checkpoint by local backend from path: checkpoints/body/pose/rtmpose-x_simcc-body7_pt-body7-halpe26_700e-384x288-7fb6e239_20230606.pth
Loads checkpoint by local backend from path: checkpoints/wholebody/rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-eaeb96c8_20230125.pth


c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmengine\utils\manager.py:113: UserWarning: <class 'mmpose.visualization.local_visualizer.PoseLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


Models initialized with EfficientTAM tracker
Pose estimation models initialized
YOLO models initialized


## Helper Functions

Define helper functions for video processing and visualization.

In [4]:
def detect_person(frame):
    """
    Detect person bounding boxes in a frame using YOLO.
    
    Args:
        frame: Input frame as numpy array
        
    Returns:
        numpy array of person bounding boxes in format [x1, y1, x2, y2]
    """
    results = model_person(frame, classes=0)  # Class 0 is person
    boxes = []
    
    for r in results:
        boxes_tensor = r.boxes.xyxy.cpu()
        confs = r.boxes.conf.cpu()
        
        for box, conf in zip(boxes_tensor, confs):
            if conf > model_person.conf:
                boxes.append(box.numpy())
    
    return np.array(boxes) if boxes else np.array([])

def visualize_wholebody_pose(frame, body_instances, left_hand_instances, right_hand_instances):
    """
    Visualize wholebody pose on a frame.
    
    Args:
        frame: Input frame
        body_instances: Body pose instances
        left_hand_instances: Left hand pose instances
        right_hand_instances: Right hand pose instances
        
    Returns:
        Frame with visualized poses
    """
    vis_frame = frame.copy()
    
    # Draw body keypoints
    if body_instances is not None:
        for i in range(len(body_instances.keypoints)):
            keypoints = body_instances.keypoints[i]
            scores = body_instances.keypoint_scores[i]
            
            # Draw body keypoints with confidence > threshold
            for j, (kp, score) in enumerate(zip(keypoints, scores)):
                if score > kpt_thresh:
                    x, y = map(int, kp)
                    cv2.circle(vis_frame, (x, y), 5, (0, 255, 0), -1)
            
            # Draw connections between keypoints (skeleton)
            # Simple skeleton connections for body
            connections = [
                (0, 1), (1, 2), (2, 3), (3, 4),  # Head to neck to shoulders
                (5, 6),  # Shoulders
                (5, 7), (7, 9),  # Left arm
                (6, 8), (8, 10),  # Right arm
                (5, 11), (6, 12),  # Shoulders to hips
                (11, 12),  # Hips
                (11, 13), (13, 15),  # Left leg
                (12, 14), (14, 16)  # Right leg
            ]
            
            for conn in connections:
                if scores[conn[0]] > kpt_thresh and scores[conn[1]] > kpt_thresh:
                    pt1 = tuple(map(int, keypoints[conn[0]]))
                    pt2 = tuple(map(int, keypoints[conn[1]]))
                    cv2.line(vis_frame, pt1, pt2, (0, 255, 0), 2)
    
    # Draw left hand keypoints
    if left_hand_instances is not None:
        for i in range(len(left_hand_instances.keypoints)):
            keypoints = left_hand_instances.keypoints[i]
            scores = left_hand_instances.keypoint_scores[i]
            
            # Draw keypoints
            for j, (kp, score) in enumerate(zip(keypoints, scores)):
                if score > kpt_thresh:
                    x, y = map(int, kp)
                    cv2.circle(vis_frame, (x, y), 3, (255, 0, 0), -1)
            
            # Hand connections
            finger_connections = [
                # Thumb
                (0, 1), (1, 2), (2, 3), (3, 4),
                # Index finger
                (0, 5), (5, 6), (6, 7), (7, 8),
                # Middle finger
                (0, 9), (9, 10), (10, 11), (11, 12),
                # Ring finger
                (0, 13), (13, 14), (14, 15), (15, 16),
                # Pinky
                (0, 17), (17, 18), (18, 19), (19, 20)
            ]
            
            for conn in finger_connections:
                if scores[conn[0]] > kpt_thresh and scores[conn[1]] > kpt_thresh:
                    pt1 = tuple(map(int, keypoints[conn[0]]))
                    pt2 = tuple(map(int, keypoints[conn[1]]))
                    cv2.line(vis_frame, pt1, pt2, (255, 0, 0), 1)
    
    # Draw right hand keypoints
    if right_hand_instances is not None:
        for i in range(len(right_hand_instances.keypoints)):
            keypoints = right_hand_instances.keypoints[i]
            scores = right_hand_instances.keypoint_scores[i]
            
            # Draw keypoints
            for j, (kp, score) in enumerate(zip(keypoints, scores)):
                if score > kpt_thresh:
                    x, y = map(int, kp)
                    cv2.circle(vis_frame, (x, y), 3, (0, 0, 255), -1)
            
            # Hand connections (same as left hand)
            finger_connections = [
                # Thumb
                (0, 1), (1, 2), (2, 3), (3, 4),
                # Index finger
                (0, 5), (5, 6), (6, 7), (7, 8),
                # Middle finger
                (0, 9), (9, 10), (10, 11), (11, 12),
                # Ring finger
                (0, 13), (13, 14), (14, 15), (15, 16),
                # Pinky
                (0, 17), (17, 18), (18, 19), (19, 20)
            ]
            
            for conn in finger_connections:
                if scores[conn[0]] > kpt_thresh and scores[conn[1]] > kpt_thresh:
                    pt1 = tuple(map(int, keypoints[conn[0]]))
                    pt2 = tuple(map(int, keypoints[conn[1]]))
                    cv2.line(vis_frame, pt1, pt2, (0, 0, 255), 1)
    
    return vis_frame

## Process Videos and Extract Wholebody Poses

Iterate through video files, detect persons, and estimate wholebody poses.

In [5]:
def process_video(video_path, output_json_path, output_video_path=None):
    """
    Process a video file to extract wholebody poses.
    
    Args:
        video_path: Path to input video file
        output_json_path: Path to save output pose data
        output_video_path: Optional path to save visualization video
    
    Returns:
        dict containing predicted wholebody poses
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video file: {video_path}")
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Initialize video writer if output path is provided
    video_writer = None
    if output_video_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(
            output_video_path,
            fourcc,
            fps,
            (width, height)
        )
    
    # Dictionary to store prediction results
    body_instances_list = []
    left_hand_instances_list = []
    right_hand_instances_list = []
    
    frame_idx = 0
    
    # Process frames
    for _ in tqdm(range(total_frames), desc=f"Processing {os.path.basename(video_path)}"):
        success, frame = cap.read()
        if not success:
            break
        
        # Detect persons in the frame
        person_boxes = detect_person(frame)
        
        # Skip if no person detected
        if len(person_boxes) == 0:
            # Create empty instances for this frame
            body_instances_list.append(dict(frame_id=frame_idx, instances=[]))
            left_hand_instances_list.append(dict(frame_id=frame_idx, instances=[]))
            right_hand_instances_list.append(dict(frame_id=frame_idx, instances=[]))
            
            # Write original frame to video if needed
            if video_writer:
                video_writer.write(frame)
                
            frame_idx += 1
            continue
        
        # Estimate wholebody pose
        body_instances, left_hand_instances, right_hand_instances = estimate_pose(
            frame, person_boxes, pose_type='wholebody', show=False
        )
        
        # Visualize poses if needed
        if show or video_writer:
            vis_frame = visualize_wholebody_pose(
                frame, body_instances, left_hand_instances, right_hand_instances
            )
            
            if show:
                cv2.namedWindow('Wholebody Pose', cv2.WINDOW_NORMAL)
                cv2.resizeWindow('Wholebody Pose', resolution[0], resolution[1])
                cv2.imshow('Wholebody Pose', vis_frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            
            if video_writer:
                video_writer.write(vis_frame)
        
        # Convert instances to serializable format
        body_instances_list.append(dict(
            frame_id=frame_idx,
            instances=split_instances(body_instances) if body_instances is not None else []
        ))
        
        left_hand_instances_list.append(dict(
            frame_id=frame_idx,
            instances=split_instances(left_hand_instances) if left_hand_instances is not None else []
        ))
        
        right_hand_instances_list.append(dict(
            frame_id=frame_idx,
            instances=split_instances(right_hand_instances) if right_hand_instances is not None else []
        ))
        
        frame_idx += 1
    
    # Release resources
    cap.release()
    if video_writer:
        video_writer.release()
    if show:
        cv2.destroyAllWindows()
    
    # Save results to JSON file
    if pose_estimator_wholebody is not None:  # Ensure model is initialized
        results = {
            'meta_info': pose_estimator_wholebody.dataset_meta,
            'body_instances': body_instances_list,
            'left_hand_instances': left_hand_instances_list,
            'right_hand_instances': right_hand_instances_list
        }
        
        with open(output_json_path, 'w') as f:
            json.dump(results, f, indent=2)
            
        print(f"Predictions saved to {output_json_path}")
        return results
    else:
        print("SAPIENS model not initialized. No results saved.")
        return None

## Process All Videos in Data Collection

Iterate through all video files in the data collection directory.

In [6]:
# Get list of video files in data directory
video_files = [f for f in os.listdir(data_dir) if f.endswith('.MP4') or f.endswith('.mp4')]

if not video_files:
    print(f"No video files found in {data_dir}")
else:
    print(f"Found {len(video_files)} video files")
    
    for video_file in video_files:
        video_name = os.path.splitext(video_file)[0]
        video_path = os.path.join(data_dir, video_file)
        
        output_json_path = os.path.join(pose_output_dir, f"{video_name}_wholebody.json")
        output_video_path = os.path.join(vis_output_dir, f"{video_name}_wholebody.mp4")
        
        # Skip if already processed
        if os.path.exists(output_json_path):
            print(f"Skipping {video_name} - already processed")
            continue
            
        print(f"Processing {video_name}...")
        try:
            process_video(video_path, output_json_path, output_video_path)
            print(f"Completed processing {video_name}")
        except Exception as e:
            print(f"Error processing {video_name}: {str(e)}")

Found 13 video files
Processing gopro10_synced_cut...


Processing gopro10_synced_cut.MP4:   0%|          | 0/60 [00:00<?, ?it/s]


0: 384x640 1 person, 56.7ms
Speed: 4.0ms preprocess, 56.7ms inference, 93.4ms postprocess per image at shape (1, 3, 384, 640)


c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmdet\models\layers\se_layer.py:158: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmdet\models\backbones\csp_darknet.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
Processing gopro10_synced_cut.MP4:   2%|▏         | 1/60 [00:01<01:14,  1.27s/it]


0: 384x640 1 person, 29.0ms
Speed: 3.0ms preprocess, 29.0ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)


...


0: 384x640 1 person, 36.3ms
Speed: 1.4ms preprocess, 36.3ms inference, 3.9ms postprocess per image at shape (1, 3, 384, 640)


Processing gopro9_synced_cut.MP4:  98%|█████████▊| 59/60 [00:09<00:00,  5.24it/s]


0: 384x640 1 person, 42.4ms
Speed: 0.0ms preprocess, 42.4ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)


Processing gopro9_synced_cut.MP4: 100%|██████████| 60/60 [00:10<00:00,  5.92it/s]


SAPIENS model not initialized. No results saved.
Completed processing gopro9_synced_cut
